# Парсинг блюд с популярных магазинов готовых блюд

### Милти

In [ ]:
import requests
from bs4 import BeautifulSoup
import re
import pandas as pd

url = 'https://www.mealty.ru/'

response = requests.get(url)
soup = BeautifulSoup(response.text, "lxml") #html.parser

data = soup.find_all("div", class_= "col-sm-4 catalog-item product_heatable heatable")
mealty = pd.DataFrame(columns = ['product_id', 'product_name', 'price', 'weight', 'calories_100g', 'proteins_100g', 'fats_100g', 'carbohydrates_100g', 'calories_portion', 'proteins_portion', 'fats_portion', 'carbohydrates_portion', 'ingredients'])

meal_id = 1
for d in data:
  product_id = meal_id

  #name
  name = d.find("div", class_="meal-card__name").text
  name_note = d.find("div", class_="meal-card__name-note").text
  product_name = name + ' ' + name_note

  #price
  price = d.find("span", class_="basket__footer-total-count green").text

  #image
  imges = d.find("div", class_ = "meal-card__image")
  url_imges = []
  for img in imges.find_all("img"):
    if img.get("class") and "newpl" in img.get("class"):
        continue
    src = img.get("data-src") or img.get("src")
    if not src:
        continue
    url_imges.append("https://www.mealty.ru" + src)

  #characteristics
  chars = d.find("div", class_ = "hidden")

  def get_float(cls):
      el = chars.find('div', class_=cls)
      text = el.get_text(strip=True).replace(',', '.') if el else '0'
      return float(text)

  #nutrients - 100g
  proteins_100g = get_float('meal-card__proteins')
  fats_100g = get_float('meal-card__fats')
  carbohydrates_100g = get_float('meal-card__carbohydrates')
  calories_100g = get_float('meal-card__calories')

  #nutrients - portion
  weight = get_float('meal-card__weight')
  calories_portion = get_float('meal-card__calories__portion')
  proteins_portion = proteins_100g*weight/100
  fats_portion = fats_100g*weight/100
  carbohydrates_portion = carbohydrates_100g*weight/100

  #ingredients
  prod_txt = chars.select_one('.meal-card__products')\
                    .get_text(' ', strip=True)



  prod = pd.DataFrame({'product_id': product_id,
                       'product_name': product_name,
                       'price': price,
                       'weight': weight,
                       'calories_100g': calories_100g,
                       'proteins_100g': proteins_100g,
                       'fats_100g': fats_100g,
                       'carbohydrates_100g': carbohydrates_100g,
                       'calories_portion': calories_portion,
                       'proteins_portion': proteins_portion,
                       'fats_portion': fats_portion,
                       'carbohydrates_portion': carbohydrates_portion,
                       'ingredients': prod_txt,
                       'company': 'Mealty'}, index=[0])

  mealty = pd.concat([mealty, prod], ignore_index=True)
  meal_id += 1


mealty

<ipython-input-4-a2cbc273b521>:79: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  mealty = pd.concat([mealty, prod], ignore_index=True)


,product_id,product_name,price,weight,calories_100g,proteins_100g,fats_100g,carbohydrates_100g,calories_portion,proteins_portion,fats_portion,carbohydrates_portion,ingredients,company
0,1,Чахохбили из курицы по-грузински с картофельны...,350,350.0,152.0,12.3,8.4,6.8,532.0,43.05,29.40,23.80,Чахохбили из курицы по-грузински [окорочок кур...,Mealty
1,2,Оджахури со свининой,350,350.0,134.0,12.5,3.6,13.0,469.0,43.75,12.60,45.50,"Картофель, свинина, лук репчатый, томаты, масл...",Mealty
2,3,Котлетка из индейки с картофельным пюре и соус...,320,340.0,177.0,8.4,10.6,11.9,602.0,28.56,36.04,40.46,"Пюре картофельное [картофель, сливки питьевые,...",Mealty
3,4,Паста в соусе Альфредо с курицей и грибами,320,350.0,228.0,9.6,15.0,13.7,798.0,33.60,52.50,47.95,Паста в соусе альфредо с грибами [спагетти отв...,Mealty
4,5,Цыпленок тандури филе с жасминовым рисом,360,350.0,215.0,11.8,10.2,19.1,753.0,41.30,35.70,66.85,"жасминовый рис [крупа рисовая, вода, масло под...",Mealty
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
62,63,Сладкая курочка в апельсиновом соусе с рисом,250,300.0,193.0,8.2,6.0,26.5,579.0,24.60,18.00,79.50,сладкая курочка в апельсиновом соусе [филе кур...,Mealty
63,64,Индейка с гречневой кашей и шампиньонами,270,350.0,143.0,9.7,5.2,14.3,501.0,33.95,18.20,50.05,гречневая каша с шампиньонами [каша гречневая ...,Mealty
64,65,Курица карри с жасминовым рисом,270,350.0,138.0,5.7,5.3,16.9,483.0,19.95,18.55,59.15,"курица карри [филе куриной грудки, вода питьев...",Mealty
65,66,Куриные тефтельки с птитимом в сливочно-грибно...,270,400.0,152.0,7.8,8.1,11.9,608.0,31.20,32.40,47.60,"куриные тефтельки [филе куриной грудки, сливки...",Mealty


In [ ]:
!pip install curl_cffi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 24.0 MB/s eta 0:00:00


In [ ]:
from curl_cffi import requests
url_vprok = 'https://www.perekrestok.ru/cat/mc/25/gotovaa-eda'
guest = requests.post(
    'https://www.perekrestok.ru/api/v1/auth/guest',
    headers={'X-Requested-With': 'XMLHttpRequest', **headers}
    user
)
token = guest.json()['token']
headers={
      'Authorization': f'Bearer {token}',
      'X-Requested-With': 'XMLHttpRequest',
      **headers}

params={'categoryId': 25, 'pageNumber': 1, 'pageSize': 100}
# Шаг 2: делаем запрос каталога с токеном


response = requests.get(url_vprok, headers = headers, params = params)
soup = BeautifulSoup(response.text, "lxml") #html.parser
product_urls = soup.find_all("a", class_ = "product-card__link")

perek = pd.DataFrame(columns = ['product_id', 'product_name', 'price', 'weight', 'calories_100g', 'proteins_100g', 'fats_100g', 'carbohydrates_100g', 'calories_portion', 'proteins_portion', 'fats_portion', 'carbohydrates_portion', 'ingredients'])

meal_id = 1
for u in product_urls:
  u = 'https://www.perekrestok.ru' + u["href"]
  response = requests.get(u, headers = headers)
  soup = BeautifulSoup(response.text, "lxml") #html.parser
  product_id = meal_id

  #name
  product_name = soup.find('h1', class_='product__title').text

  #price
  raw_text = soup.find('div', class_='price-new').get_text()
  price_match = re.search(r'[\d]+,[\d]+', raw_text)
  price = price_match.group().replace(',', '.') if price_match else None
  price = float(price)

  #image
  image_url = soup.find('img', class_='img-top-slider')['src']

  #characteristics
  chars = soup.find_all('div', class_='product-calories-item__value')

  #nutrients - 100g
  proteins_100g = float(chars[1].replace('г', '').strip().replace(',', '.'))
  fats_100g = float(chars[2].replace('г', '').strip().replace(',', '.'))
  carbohydrates_100g = float(chars[3].replace('г', '').strip().replace(',', '.'))
  calories_100g = float(chars[0].replace('г', '').strip().replace(',', '.'))

  #nutrients - portion
  match = re.search(r'(\d+)\s*г', product_name)
  weight = float(match.group(1)) if match else None
  calories_portion = calories_100g*weight/100
  proteins_portion = proteins_100g*weight/100
  fats_portion = fats_100g*weight/100
  carbohydrates_portion = carbohydrates_100g*weight/100

  #ingredients
  prod_txt = soup.find('div', class_="sc-dlfnuX fCHXmV").get_text()



  prod = pd.DataFrame({'product_id': product_id,
                       'product_name': product_name,
                       'price': price,
                       'weight': weight,
                       'calories_100g': calories_100g,
                       'proteins_100g': proteins_100g,
                       'fats_100g': fats_100g,
                       'carbohydrates_100g': carbohydrates_100g,
                       'calories_portion': calories_portion,
                       'proteins_portion': proteins_portion,
                       'fats_portion': fats_portion,
                       'carbohydrates_portion': carbohydrates_portion,
                       'ingredients': prod_txt,
                       'company': 'Perekrestok'}, index=[0])

  perek = pd.concat([perek, prod], ignore_index=True)
  meal_id += 1


perek
print(response.status_code)

JSONDecodeError: unexpected character: line 1 column 1 (char 0)

In [ ]:
import requests
from bs4 import BeautifulSoup
import json
import re


# Задаем User-Agent для имитации реального браузера
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/134.0.0.0 YaBrowser/25.4.0.0 Safari/537.36'
}

BASE_URL = "https://vkusvill.ru"


# Функция для получения ссылок на категории
def get_categories():
    response = requests.get(f"{BASE_URL}/goods/", headers=HEADERS)
    soup = BeautifulSoup(response.text, "html.parser")

    # Находим блоки с категориями по классу
    categories = []
    for link in soup.select(".VVCatalog2020Menu__List a"):
        categories.append({
            "name": link.get_text(strip=True),
            "url": BASE_URL + link["href"]
        })
    return categories


# Функция для получения количества страниц в категории
def get_total_pages(category_url):
    response = requests.get(category_url, headers=HEADERS)
    soup = BeautifulSoup(response.text, "html.parser")

    # Ищем кнопку последней страницы
    last_page = soup.select(".VV_Pager.js-lk-pager a")
    if len(last_page) > 1:
        return int(last_page[-2].get("data-page"))
    else:
        return 1


# Функция для парсинга продуктов на странице категории
def parse_products_in_category(category_url):
    total_pages = get_total_pages(category_url)
    products = []

    # Проходим по каждой странице категории
    for page in range(1, total_pages + 1):
        page_url = f"{category_url}?PAGEN_1={page}"
        response = requests.get(page_url, headers=HEADERS)
        soup = BeautifulSoup(response.text, 'html.parser')

        # Парсим карточки товаров на странице
        product_cards = soup.select('.ProductCards__list .ProductCard')

        for card in product_cards:
            name_element = card.select_one('.ProductCard__link')
            link = ''
            if name_element:
                name = re.sub(r'[^a-zA-Z0-9а-яА-ЯёЁ]', ' ', name_element.get('title').strip())
                link = BASE_URL + name_element['href']
            quantity = card.select_one('.ProductCard__weight')
            if quantity:
                quantity = re.sub(r'[^a-zA-Z0-9а-яА-ЯёЁ]', ' ', quantity.get_text(strip=True))
            price = card.select_one('.Price.Price--md.Price--gray.Price--label')
            if price:
                price = int(re.sub(r'[^0-9]', '', price.get_text(strip=True)))

            # Добавляем информацию о продукте в список
            products.append({
                "name": name,
                "link": link,
                "quantity": quantity,
                "price": price
            })
    return products


# Основная функция для парсинга всех категорий и записи в JSON
good_category = {
    "Овощи, фрукты, ягоды, зелень",
    "Хлеб и выпечка",
    "Выпекаем сами",
    "Молочные продукты, яйцо",
    "Мясо, птица",
    "Рыба, икра и морепродукты",
    "Колбаса, сосиски, деликатесы",
    "Замороженные продукты",
    "Сыры",
    "Напитки",
    "Орехи, чипсы и снеки",
    "Вегетарианское и постное",
    "Крупы, макароны, мука",
    "Алкоголь",
    "Консервация",
    "Чай и кофе",
    "Масла, соусы, специи, сахар и соль"
}


def parse_product_from_vv():
    categories = get_categories()
    data = {}

    for category in categories:
        if category["name"] not in good_category:
            continue
        print(f"Парсим категорию: {category['name']}", end='... ')
        try:
            products = parse_products_in_category(category["url"])
            data[category['name']] = products
            print(f"обработано {len(products)} продуктов")
        except:
            print("Не получилось обработать")

    with open(BD_path, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=4)

    print(f"Все продукты сохранены во {BD_path}")

parse_product_from_vv()

Парсим категорию: Овощи, фрукты, ягоды, зелень... обработано 785 продуктов
Парсим категорию: Хлеб и выпечка... обработано 456 продуктов
Парсим категорию: Выпекаем сами... обработано 60 продуктов
Парсим категорию: Молочные продукты, яйцо... обработано 531 продуктов
Парсим категорию: Мясо, птица... обработано 418 продуктов
Парсим категорию: Рыба, икра и морепродукты... обработано 563 продуктов
Парсим категорию: Колбаса, сосиски, деликатесы... обработано 290 продуктов
Парсим категорию: Замороженные продукты... обработано 1094 продуктов
Парсим категорию: Сыры... обработано 238 продуктов
Парсим категорию: Напитки... обработано 613 продуктов
Парсим категорию: Орехи, чипсы и снеки... обработано 485 продуктов
Парсим категорию: Вегетарианское и постное... обработано 2621 продуктов
Парсим категорию: Крупы, макароны, мука... обработано 245 продуктов
Парсим категорию: Алкоголь... Не получилось обработать
Парсим категорию: Консервация... обработано 451 продуктов
Парсим категорию: Чай и кофе... обра

NameError: name 'BD_path' is not defined

# Функции расчета нормы КБЖУ для подбора рациона


In [ ]:
def calculate_bmr(gender: str, weight: float, height: float, age: int) -> float:
    """
    Расчёт базального обмена веществ (BMR) по формуле Миффлина – Сан Жеора.

    :param gender: 'male' или 'female'
    :param weight: Вес в килограммах
    :param height: Рост в сантиметрах
    :param age: Возраст в годах
    :return: BMR в килокалориях
    """
    if gender.lower() == 'male':
        return 10 * weight + 6.25 * height - 5 * age + 5
    elif gender.lower() == 'female':
        return 10 * weight + 6.25 * height - 5 * age - 161
    else:
        raise ValueError("Пол должен быть 'male' или 'female'.")

def get_activity_factor(activity_level: str) -> float:
    """
    Получение коэффициента активности на основе уровня физической активности.

    :param activity_level: Уровень активности ('sedentary', 'light', 'moderate', 'high', 'very_high')
    :return: Коэффициент активности
    """
    activity_levels = {
        'sedentary': 1.2,
        'light': 1.375,
        'moderate': 1.55,
        'high': 1.725,
        'very_high': 1.9
    }
    if activity_level not in activity_levels:
        raise ValueError("Недопустимый уровень активности.")
    return activity_levels[activity_level]

def get_macronutrient_factors(activity_level: str) -> tuple:
    """
    Получение коэффициентов для расчёта белков и жиров на кг массы тела.

    :param activity_level: Уровень активности ('sedentary', 'light', 'moderate', 'high', 'very_high')
    :return: Кортеж с коэффициентами (p, f)
    """
    factors = {
        'sedentary': (1.2, 0.8),
        'light': (1.3, 0.9),
        'moderate': (1.5, 1.0),
        'high': (1.7, 1.1),
        'very_high': (2.0, 1.2)
    }
    if activity_level not in factors:
        raise ValueError("Недопустимый уровень активности.")
    return factors[activity_level]

def calculate_tdee(bmr: float, activity_factor: float) -> float:
    """
    Расчёт общего суточного расхода энергии (TDEE).

    :param bmr: Базальный обмен веществ
    :param activity_factor: Коэффициент активности
    :return: TDEE в килокалориях
    """
    return bmr * activity_factor

def calculate_macros(weight: float, tdee: float, p: float, f: float) -> dict:
    """
    Расчёт суточной нормы макронутриентов в граммах.

    :param weight: Вес в килограммах
    :param tdee: Общий суточный расход энергии
    :param p: Коэффициент белка на кг массы тела
    :param f: Коэффициент жира на кг массы тела
    :return: Словарь с количеством белков, жиров и углеводов в граммах
    """
    protein_grams = p * weight
    fat_grams = f * weight
    protein_calories = protein_grams * 4
    fat_calories = fat_grams * 9
    carb_calories = tdee - (protein_calories + fat_calories)
    if carb_calories < 0:
        raise ValueError("Недостаточно калорий для покрытия потребностей в белках и жирах.")
    carb_grams = carb_calories / 4
    return {
        'protein_grams': round(protein_grams, 2),
        'fat_grams': round(fat_grams, 2),
        'carb_grams': round(carb_grams, 2)
    }

def calculate_nutrition(gender: str, age: int, weight: float, height: float, activity_level: str) -> dict:
    """
    Основная функция для расчёта BMR, TDEE и макронутриентов.

    :param gender: 'male' или 'female'
    :param age: Возраст в годах
    :param weight: Вес в килограммах
    :param height: Рост в сантиметрах
    :param activity_level: Уровень активности ('sedentary', 'light', 'moderate', 'high', 'very_high')
    :return: Словарь с результатами расчётов
    """
    bmr = calculate_bmr(gender, weight, height, age)
    activity_factor = get_activity_factor(activity_level)
    tdee = calculate_tdee(bmr, activity_factor)
    p, f = get_macronutrient_factors(activity_level)
    macros = calculate_macros(weight, tdee, p, f)
    return {
        'BMR': round(bmr, 2),
        'TDEE': round(tdee, 2),
        'Protein (g)': macros['protein_grams'],
        'Fat (g)': macros['fat_grams'],
        'Carbohydrates (g)': macros['carb_grams']
    }

# Пример использования:

gender = 'male'
age = 20
weight = 83  # кг
height = 187  # см
activity_level = 'moderate'  # 'sedentary', 'light', 'moderate', 'high', 'very_high'


nutrition = calculate_nutrition(gender, age, weight, height, activity_level)
for key, value in nutrition.items():
  print(f"{key}: {value}")

BMR: 1903.75
TDEE: 2950.81
Protein (g): 124.5
Fat (g): 83.0
Carbohydrates (g): 426.45
